# 01 - Cache cliff

Scan of `max` over a `u8` array vs `int64` for increasing sizes. The gain spikes when the `u8` array still fits in cache but `int64` (8x larger) has already overflowed it. **All measured on this machine** (run the cells).

In [ ]:
import os, subprocess, tempfile
import matplotlib.pyplot as plt

def find_include():
    d = os.getcwd()
    for _ in range(6):
        cand = os.path.join(d, "include", "smart2raw.h")
        if os.path.exists(cand):
            return os.path.join(d, "include")
        d = os.path.dirname(d)
    raise FileNotFoundError("include/smart2raw.h not found; run from the repo")

INC = find_include()

def compile_run(src, args_list, flags="-O3 -march=native"):
    """Compiles a C harness (with the header) and runs it for each args; returns outputs."""
    with tempfile.NamedTemporaryFile("w", suffix=".c", delete=False) as f:
        f.write(src); cpath = f.name
    exe = cpath[:-2]
    subprocess.check_call(["gcc"] + flags.split() + ["-I", INC, "-o", exe, cpath])
    outs = []
    for a in args_list:
        outs.append(subprocess.check_output([exe] + [str(x) for x in a]).decode().strip())
    os.remove(cpath); os.remove(exe)
    return outs

print("include:", INC)

## Measurement (compiles a C harness and sweeps N)

In [ ]:
SRC = r"""#define _POSIX_C_SOURCE 199309L
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <time.h>
static double ms(){struct timespec t;clock_gettime(CLOCK_MONOTONIC,&t);return t.tv_sec*1e3+t.tv_nsec*1e-6;}
int main(int argc,char**argv){
  size_t N=argc>1?strtoull(argv[1],0,10):100000;
  uint8_t *a=malloc(N); int64_t *b=malloc(N*8);
  for(size_t i=0;i<N;i++){a[i]=(uint8_t)(i*131);b[i]=(int64_t)a[i];}
  double bu=1e18,bi=1e18; volatile uint64_t s=0;
  for(int r=0;r<9;r++){double t=ms();uint8_t m=0;for(size_t i=0;i<N;i++)if(a[i]>m)m=a[i];s+=m;double d=ms()-t;if(d<bu)bu=d;}
  for(int r=0;r<9;r++){double t=ms();int64_t m=0;for(size_t i=0;i<N;i++)if(b[i]>m)m=b[i];s+=(uint64_t)m;double d=ms()-t;if(d<bi)bi=d;}
  printf("%zu %.6f %.6f\n",N,bu,bi);free(a);free(b);(void)s;return 0;}"""

sizes = [2000,4000,8000,16000,32000,64000,128000,256000,512000,
         1000000,2000000,4000000,8000000,16000000,32000000]
rows = [list(map(float, o.split())) for o in compile_run(SRC, [[n] for n in sizes])]
N   = [r[0] for r in rows]
spd = [r[2]/r[1] for r in rows]   # int64_time / u8_time
for n,s in zip(N,spd): print(f"N={int(n):>9}  speedup={s:5.1f}x")

## Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.semilogx(N, spd, "o-", color="#2E5A88", lw=2)
for x, lab in [(48*1024,"L1 48KB"), (2*1024*1024,"L2 2MB")]:
    ax.axvline(x, color="#C2772E", ls=":")
    ax.text(x, max(spd)*0.95, lab, rotation=90, va="top", color="#C2772E", fontsize=8)
ax.set_xlabel("N (elements; array u8 = N bytes, int64 = 8N bytes)")
ax.set_ylabel("speedup (max scan: int64 / u8)")
ax.set_title("Cache cliff (measured on this machine)")
ax.grid(alpha=0.3); plt.show()

Expected: a peak in the L1/L2 window (where u8 fits and int64 does not) and a smaller plateau in RAM. Honest note: this holds for scans (max/min/count); the *naive* byte sum only gains with the SIMD kernel `vpsadbw` (see `sum_fast`).